# GEMV resource report

For K=1536,N=8960, compare 1/4/8 weight rows per program, with four warps. Report registers per thread, Triton spill metadata, shared bytes, grid size, padding, device and source identity. Each candidate passes an FP32 accuracy check before reporting. This does not measure occupancy or latency.


In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/fastinfer')
python = Path('/kaggle/working/fastinfer-t4-py312/bin/python')
assert python.exists(), 'The isolated Python 3.12 GPU environment is required.'
assert (repo / 'kernels/triton_gemv.py').exists(), 'Existing GEMV kernel checkout required.'

source = '"""Report compiled GEMV resources; this script does not measure latency."""\n\nimport argparse\nimport hashlib\nimport json\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef positive_int(value: str) -> int:\n    result = int(value)\n    if result <= 0:\n        raise argparse.ArgumentTypeError("must be a positive integer")\n    return result\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--k", type=positive_int, default=1536)\n    parser.add_argument("--n", type=positive_int, default=8960)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    if args.k > 16384:\n        parser.error("K must be at most 16384, matching the GEMV wrapper")\n\n    import torch\n    import triton\n\n    from fastinfer.kernels.triton_gemv import GEMV_CONFIGS, _gemv\n\n    if not torch.cuda.is_available():\n        parser.error("a CUDA GPU is required")\n\n    torch.manual_seed(0)\n    torch.set_float32_matmul_precision("highest")\n    device = torch.cuda.current_device()\n    pad_k = triton.next_power_of_2(args.k)\n    variants = []\n\n    with torch.inference_mode():\n        a = torch.randn((1, args.k), device="cuda", dtype=torch.float16)\n        weight = torch.randn((args.n, args.k), device="cuda", dtype=torch.float16)\n        expected = a.float() @ weight.float().t()\n\n        for name, config in GEMV_CONFIGS.items():\n            rows = config["block_rows"]\n            grid = (triton.cdiv(args.n, rows),)\n            output = torch.empty((1, args.n), device=a.device, dtype=a.dtype)\n\n            # Same kernel and launch arguments as gemv(); retain its compiled handle.\n            compiled = _gemv[grid](\n                a,\n                weight,\n                output,\n                args.n,\n                args.k,\n                rows,\n                pad_k,\n                num_warps=config["num_warps"],\n            )\n            torch.cuda.synchronize()\n            torch.testing.assert_close(output.float(), expected, rtol=.01, atol=.01)\n            difference = (output.float() - expected).abs()\n\n            variants.append({\n                "name": name,\n                "block_rows": rows,\n                "pad_k": pad_k,\n                "programs": grid[0],\n                "warps_per_program": compiled.metadata.num_warps,\n                "threads_per_program": 32 * compiled.metadata.num_warps,\n                "registers_per_thread": compiled.n_regs,\n                "spills": compiled.n_spills,\n                "shared_memory_bytes_per_program": compiled.metadata.shared,\n                "correctness": {\n                    "reference": "pytorch_float32",\n                    "rtol": .01,\n                    "atol": .01,\n                    "passed": True,\n                    "max_abs_error": difference.max().item(),\n                },\n            })\n\n    environment = json.loads(subprocess.check_output(\n        [sys.executable, str(ROOT / "scripts/environment_report.py")],\n        cwd=ROOT,\n        text=True,\n    ))\n    report = {\n        "diagnostic": "gemv_compiled_resources",\n        "environment": environment,\n        "device_index": device,\n        "gpu": torch.cuda.get_device_name(device),\n        "compute_capability": list(torch.cuda.get_device_capability(device)),\n        "sm_count": torch.cuda.get_device_properties(device).multi_processor_count,\n        "shape": {"m": 1, "k": args.k, "n": args.n},\n        "dtype": "float16",\n        "strides": {"a": list(a.stride()), "weight": list(weight.stride())},\n        "kernel_sha256": hashlib.sha256(\n            (ROOT / "kernels/triton_gemv.py").read_bytes()\n        ).hexdigest(),\n        "interpretation": (\n            "Resource metadata from the compiled kernel, not measured occupancy or latency. "\n            "spills is Triton\'s n_spills field, not a count of runtime memory transactions. "\n            "Zero spills does not establish full occupancy or identify a bottleneck."\n        ),\n        "variants": variants,\n    }\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n    destination = args.output or ROOT / "results/raw" / f"gemv_resources_{stamp}.json"\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(json.dumps(report, indent=2) + "\\n")\n\n    print(f"GPU: {report[\'gpu\']} | SMs: {report[\'sm_count\']} | K={args.k} | PAD_K={pad_k}")\n    print("Rows/program | Programs | Threads/program | Registers/thread | Spills | Shared bytes | FP32 check")\n    for item in variants:\n        print(\n            f"{item[\'block_rows\']:12} | {item[\'programs\']:8} | "\n            f"{item[\'threads_per_program\']:15} | {item[\'registers_per_thread\']:16} | "\n            f"{item[\'spills\']:6} | {item[\'shared_memory_bytes_per_program\']:12} | PASS"\n        )\n    print(report["interpretation"])\n    print(f"Saved: {destination}")\n\n\nif __name__ == "__main__":\n    main()\n'
script = repo / 'scripts/gemv_kernel_report.py'
script.write_text(source)
subprocess.run([str(python), str(script)], cwd=repo, check=True)
